# Dubai Real Estate Analysis: Python Notebook

This notebook covers the Python part of the project:

1. **Setup:** folders and database settings
2. **Load into MySQL:** load the five CSV files into the tables created by `sql/01_create_tables.sql`
3. **Read the CSV files:** for cleaning and analysis
4. **Clean the data:** dates, repeated columns and duplicate checks
5. **Metro effect:** compare near and far listings *within each community*
6. **Save the clean files:** used by the Power BI dashboard

**Before you run it:** create the tables in MySQL first (`sql/01_create_tables.sql`), and install the packages with
`pip install pandas sqlalchemy pymysql`.

## 1. Setup

Change `DATA_FOLDER` to the folder that holds the five CSV files, and `MYSQL_USER` if you don't use `root`.
The MySQL password is **not stored** in this notebook. You will be asked for it when you run section 2.

In [ ]:
import pandas as pd
from pathlib import Path
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# ---- Settings you may need to change ----
DATA_FOLDER = Path(r"C:\path\to\your\data")      # folder with the 5 CSV files
OUTPUT_FOLDER = DATA_FOLDER.parent / "clean"        # cleaned files are saved here
MYSQL_USER = "root"
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DATABASE = "dubai_real_estate"
# ------------------------------------------

OUTPUT_FOLDER.mkdir(exist_ok=True)

# Expected number of rows in each table (from the dataset description)
EXPECTED_ROWS = {
    "secondary_sales": 50_000,
    "off_plan": 12_000,
    "rentals": 25_000,
    "area_prices_monthly": 6_384,
    "metro_stations": 55,
}

## 2. Load the CSV files into MySQL

**Why Python and not the Workbench import wizard?** The wizard was very slow on the 50,000-row file, kept running in the background,
and silently dropped the 760 rows with a blank off-plan price. Loading with pandas keeps blanks as `NULL`.

**Safety check:** a table is only loaded if it is **empty**. Running the loader twice once gave 100,000 rows for 50,000 unique listings,
so this check prevents duplicates.

True/False columns are stored as text (`'True'` / `'False'`) because the tables were created that way.

In [ ]:
password = getpass("MySQL password: ")      # typed when you run the cell, never saved in the file

engine = create_engine(URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=password,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE,
))

def load_table(table):
    """Load one CSV into its MySQL table, but only if the table is empty."""
    with engine.connect() as conn:
        existing = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
    if existing > 0:
        print(f"{table}: already has {existing:,} rows, skipped")
        return

    df = pd.read_csv(DATA_FOLDER / f"{table}.csv")

    # Store True/False columns as text, as defined in the CREATE TABLE statements
    for col in df.select_dtypes(include="bool").columns:
        df[col] = df[col].astype(str)

    # Floor numbers are blank for villas: keep them as whole numbers with NULLs
    for col in ("floor", "total_floors"):
        if col in df.columns:
            df[col] = df[col].astype("Int64")

    df.to_sql(table, engine, if_exists="append", index=False, chunksize=5000, method="multi")
    print(f"{table}: {len(df):,} rows loaded")

for table in EXPECTED_ROWS:
    load_table(table)

### Check the load

Row counts should match the expected numbers, and listing IDs should be unique.

In [ ]:
with engine.connect() as conn:
    for table, expected in EXPECTED_ROWS.items():
        n = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        status = "OK" if n == expected else f"CHECK (expected {expected:,})"
        print(f"{table:<22}{n:>8,}  {status}")

    for table in ("secondary_sales", "off_plan", "rentals"):
        n = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        unique = conn.execute(text(f"SELECT COUNT(DISTINCT id) FROM {table}")).scalar()
        print(f"{table}: {n:,} rows, {unique:,} unique ids", "OK" if n == unique else "DUPLICATES")

## 3. Read the CSV files for cleaning and analysis

In [ ]:
sales   = pd.read_csv(DATA_FOLDER / "secondary_sales.csv")
offplan = pd.read_csv(DATA_FOLDER / "off_plan.csv")
rentals = pd.read_csv(DATA_FOLDER / "rentals.csv")
prices  = pd.read_csv(DATA_FOLDER / "area_prices_monthly.csv")
metro   = pd.read_csv(DATA_FOLDER / "metro_stations.csv")

for name, df in [("secondary_sales", sales), ("off_plan", offplan), ("rentals", rentals),
                 ("area_prices_monthly", prices), ("metro_stations", metro)]:
    print(f"{name:<22}{df.shape[0]:>8,} rows  {df.shape[1]:>3} columns")

## 4. Clean the data

Steps:
1. Convert text dates into real dates.
2. Drop the square-metre columns: they only repeat the square-foot columns.
3. Check for duplicate listing IDs (every count should be `0`).
4. Look at the blank values. Blank floor numbers (villas) and blank off-plan prices (months with no off-plan listings) are real, so they are **kept as blanks, not filled with 0**.

In [ ]:
# 1. Dates
for df in (sales, offplan, rentals):
    df["date_listed"] = pd.to_datetime(df["date_listed"])
prices["year_month"] = pd.to_datetime(prices["year_month"])

# 2. Drop repeated columns
sales   = sales.drop(columns=["area_m2", "price_per_m2_usd"])
offplan = offplan.drop(columns=["area_m2", "price_per_m2_usd"])
rentals = rentals.drop(columns=["area_m2", "rent_per_m2_usd"])
prices  = prices.drop(columns=["secondary_price_per_m2_usd"])

# 3. Duplicate ids (should all be 0)
print("Duplicate ids  ->  sales:", sales.duplicated("id").sum(),
      "| off-plan:", offplan.duplicated("id").sum(),
      "| rentals:", rentals.duplicated("id").sum())

# 4. Blank values worth knowing about
print("Blank floor numbers (secondary_sales):", sales["floor"].isna().sum())
print("Blank off-plan prices (area_prices_monthly):", prices["offplan_price_per_sqft_usd"].isna().sum())

### Metro distance buckets

`metro_distance_min` is the minutes to the nearest station. Listings are grouped into four buckets: **0-5, 6-10, 11-20 and 20+** minutes.

Remember that the minutes are either **walking** or **driving** (`metro_distance_type`), and only about 14% of listings are walking distance.

In [ ]:
sales["metro_bucket"] = pd.cut(
    sales["metro_distance_min"],
    bins=[0, 5, 10, 20, float("inf")],
    labels=["0-5", "6-10", "11-20", "20+"],
)

print(sales["metro_distance_type"].value_counts())
print(sales.groupby("metro_distance_type")["metro_bucket"].value_counts().unstack())

## 5. Does being near a metro station raise the price?

**The problem:** comparing all near listings with all far listings gave the opposite answer (apartments near stations looked *cheaper*), because stations are in cheaper communities. That is a **community mix** effect, not a metro effect.

**The fix:** compare near and far listings **inside each community**, one community at a time, so every comparison is like with like.

- Only **walking** listings are used, because walking and driving minutes cannot be compared.
- **Near** means 10 minutes or less on foot. **Far** is more than 10 minutes.
- Price per sq ft = total price / total area (this weights large and small units fairly).
- Only communities with both near and far walking listings can be compared.

In [ ]:
walk = sales[sales["metro_distance_type"] == "walk"].copy()
walk["near"] = walk["metro_distance_min"] <= 10

totals = walk.groupby(["community", "near"])[["price_usd", "area_sqft"]].sum()
totals["price_per_sqft"] = totals["price_usd"] / totals["area_sqft"]

by_community = (
    totals["price_per_sqft"].unstack()      # columns: False (far) and True (near)
    .dropna()                               # keep communities that have both groups
    .rename(columns={False: "far", True: "near"})
)
by_community["near_vs_far_pct"] = (by_community["near"] / by_community["far"] - 1) * 100

print(f"{len(by_community)} communities compared\n")
print(by_community.round(1))
print("\nMedian near-vs-far difference: {:.1f}%".format(by_community["near_vs_far_pct"].median()))
print("Share of communities where near is pricier: {:.0%}".format((by_community["near_vs_far_pct"] > 0).mean()))

**How to read the result:** the median difference is the number used in the README. A positive value means listings within a 10-minute walk of a station
are priced higher than similar listings further away in the same community.

**Limits:** the sample is small (only a handful of communities have both groups), and the test does not control for property size, age or type.
It is an indication, not proof.

## 6. Save the clean files for Power BI

In [ ]:
sales.to_csv(OUTPUT_FOLDER / "secondary_sales_clean.csv", index=False)
offplan.to_csv(OUTPUT_FOLDER / "off_plan_clean.csv", index=False)
rentals.to_csv(OUTPUT_FOLDER / "rentals_clean.csv", index=False)
prices.to_csv(OUTPUT_FOLDER / "area_prices_monthly_clean.csv", index=False)
metro.to_csv(OUTPUT_FOLDER / "metro_stations_clean.csv", index=False)

print("Clean files saved to", OUTPUT_FOLDER)